# 06 · Inference test (load best model → image → JSON)
Loads the **best checkpoint** + the **config snapshot** saved during training and predicts JSON for an image you choose. Decoding is **greedy** (deterministic) — sampling would only add hallucination, the opposite of faithful extraction.

In [ ]:
# --- Bootstrap: make the package importable without installing, and stay OFFLINE.
import os, sys
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
print("repo root:", ROOT)


In [ ]:
from pathlib import Path
from gemma_ft_json.config import load_config
from gemma_ft_json.inference import Predictor

cfg = load_config(ROOT / 'configs' / 'default.yaml')
run_dir = ROOT / 'runs' / cfg.project.name
snapshot = run_dir / 'config.snapshot.yaml'
checkpoint = run_dir / 'best.ckpt'
assert snapshot.is_file() and checkpoint.is_file(), 'Train first (notebook 04).'
predictor = Predictor(snapshot, checkpoint, max_new_tokens=256)
print(predictor.describe())

### Choose an image
Set `IMAGE_PATH` to your own file, **or** use the file-upload widget below (works in classic Jupyter; in other front-ends just set the path manually).

In [ ]:
# Option A: hard-code a path (default: first demo image).
demo_imgs = sorted((ROOT / 'data' / 'demo' / 'images').glob('*.png'))
IMAGE_PATH = str(demo_imgs[0]) if demo_imgs else None

# Option B: interactive upload (uncomment in classic Jupyter).
# import ipywidgets as widgets
# up = widgets.FileUpload(accept='image/*', multiple=False); display(up)
# # after uploading: write the bytes to a temp file and set IMAGE_PATH to it.
print('IMAGE_PATH =', IMAGE_PATH)

In [ ]:
import json
from PIL import Image
import matplotlib.pyplot as plt

assert IMAGE_PATH, 'Set IMAGE_PATH to an image file.'
pred = predictor.predict(IMAGE_PATH)
fig, ax = plt.subplots(1, 1, figsize=(6, 6))
ax.imshow(Image.open(IMAGE_PATH)); ax.axis('off'); ax.set_title('input image'); plt.show()
print('=== RAW MODEL OUTPUT ===')
print(pred)
try:
    print('\n=== PARSED JSON ===')
    print(json.dumps(json.loads(pred), indent=2))
except Exception as e:
    print('\n(Not valid JSON yet — expected for a small/under-trained demo model.)', e)

On a real fine-tune (Gemma 3 270M decoder + your data) the raw output should parse as JSON matching the table. The demo `stub` model emits gibberish — that is expected.